- 피처 엔지니어링(Feature Engineering) : 원시 데이터(Raw Data)를 머신러닝 모델이 더 잘 이해하고 학습할 수 있는 형태로 변환하거나 새로운 특징(Feature)을 만들어내는 과정

1. 결측치 처리 (Imputation)

In [16]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
# 예제 데이터 생성
df = pd.DataFrame({'Age': [25, np.nan, 30, 22, np.nan], 'City': ['Seoul', 'Busan', np.nan, 'Seoul', 'Jeju']})

# 방법 1: pandas를 이용한 결측치 채우기 (연령은 중앙값, 도시는 최빈값)
df['Age'] = df['Age'].fillna(df.Age.median())
df['City'] = df['City'].fillna(df.City.mode()[0])

# 방법 2: scikit-learn의 SimpleImputer 사용
imputer = SimpleImputer(strategy='median')  
df[['Age']] = imputer.fit_transform(df[['Age']])

- 수치 데이터에 대한 결측치 처리는 평균, 중앙값을 이용할 수 있는데 시계열 데이터는 이전 또는 이후 데이터로 채우는 것이 일반적
- 범주형 데이터(종류가 한정되어 있는 문자열)는 최빈값을 이용해서 결측치를 처리하는 것이 일반적

In [17]:
df

,Age,City
0,25.0,Seoul
1,25.0,Busan
2,30.0,Seoul
3,22.0,Seoul
4,25.0,Jeju


2. 범주형 변수 인코딩 (Categorical Encoding)
   - 컴퓨터는 텍스트를 이해하지 못하므로, '성별'이나 '도시' 같은 문자열 형태의 범주형 데이터를 숫자로 변환해야 함.

   - 종류 :
     - 원-핫 인코딩 (One-Hot Encoding): 순서가 없는 명목형 데이터에 사용. 각 카테고리를 새로운 열로 만들고 0과 1로 표시
     - 레이블 인코딩 (Label Encoding): 순서가 있는 데이터(예: 등급 - 낮음, 중간, 높음)를 0, 1, 2 등의 숫자로 맵핑

In [28]:
from sklearn.preprocessing import LabelEncoder

# 예제 데이터 생성
df = pd.DataFrame({'Color': ['Blue','Red' , 'Green', 'Brown'], 'Size': ['S', 'M', 'L','XL']})

# 1. 원-핫 인코딩 (pandas get_dummies 활용)
# 순서가 없는 'Color'에 적용. 
df_onehot = pd.get_dummies( df, columns=['Color'], drop_first=True)

# 2. 레이블 인코딩 (scikit-learn LabelEncoder 활용)
# 트리 기반 모델에 주로 사용
le = LabelEncoder()
df['Color_Label'] = le.fit_transform( df['Color'])

# 순서가 있는 'Size'는 매핑 딕셔너리를 사용하는 것이 안전
size_mapping = {'S':0, 'M':1, 'L':2,'XL':5}
df['Size_mapped'] = df['Size'].map( size_mapping )

In [29]:
df

,Color,Size,Color_Label,Size_mapped
0,Blue,S,0,0
1,Red,M,3,1
2,Green,L,2,2
3,Brown,XL,1,5


In [30]:
df_onehot

,Size,Color_Brown,Color_Green,Color_Red
0,S,False,False,False
1,M,False,False,True
2,L,False,True,False
3,XL,True,False,False


3. 수치형 변수 스케일링 (Feature Scaling)
   - 서로 다른 피처(예: 10,000 단위의 '연봉'과 1~100 단위의 '나이')의 단위나 범위가 크게 다를 경우, 모델이 범위가 큰 변수에 과도한 가중치를 부여할 수 있어 이를 맞춰주는 작업
   - 종류 :
        - 표준화 (Standardization): 데이터의 평균을 0, 분산을 1로 맞춤 (정규분포를 따를 때 유용)
        - 정규화 (Normalization - MinMax): 데이터의 값을 0과 1 사이로 압축

In [37]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 예제 데이터 생성
df = pd.DataFrame({'Salary': [50000000, 60000000, 45000000], 'Age': [25, 45, 30]})

# 1. 표준화 (StandardScaler)
std_scaler = StandardScaler()
df_std = pd.DataFrame( std_scaler.fit_transform( df ) , columns=df.columns)

# 2. 정규화 (MinMaxScaler)

minmax_scaler = MinMaxScaler()
df_minmax =  pd.DataFrame( minmax_scaler.fit_transform( df ) , columns=df.columns )


In [38]:
df_minmax

,Salary,Age
0,0.333333,0.00
1,1.000000,1.00
2,0.000000,0.25


In [36]:
df_std

,Salary,Age
0,-0.267261,-0.980581
1,1.336306,1.372813
2,-1.069045,-0.392232


4. 데이터 변환 및 구간화 (Transformation & Binning)
  - 데이터의 분포를 모델이 학습하기 좋은 형태로 바꾸거나, 연속형 수치를 범주형으로 바꾸는 작업입니다.

  - 종류:
     - 로그 변환 (Log Transformation): 데이터가 한쪽으로 심하게 치우쳐 있을 때(Skewed) 정규분포에 가깝게 펴주는 역할
     - 구간화 (Binning): 연속형 데이터를 특정 구간으로 나누어 범주형으로 만듬 (예: 25세 -> '20대')

In [42]:
df['Log_Salary'] = np.log1p(df.Salary)

In [43]:
df

,Salary,Age,Log_Salary
0,50000000,25,17.727534
1,60000000,45,17.909855
2,45000000,30,17.622173


In [46]:
# 2. 구간화 (Binning)
df_age = pd.DataFrame({'Age':[100,20,2,45,60,45,77]})
bins = [0,19,39,59,100]
labels = [ 'Teen', 'Young Adult', 'Adult', 'Senior']
df_age['Age_Group'] = pd.cut( df_age['Age'] , bins=bins , labels= labels )

In [47]:
df_age

,Age,Age_Group
0,100,Senior
1,20,Young Adult
2,2,Teen
3,45,Adult
4,60,Senior
5,45,Adult
6,77,Senior


5. 파생 변수 생성 (Feature Creation)
  - 기존의 데이터를 조합하거나 분해해서 모델 예측에 도움이 되는 완전히 새로운 특징을 만들어냄. 비즈니스 도메인 지식이 가장 많이 요구되는 부분
  - 방법: 날짜 데이터 분해, 두 변수의 사칙연산, 다항식(Polynomial) 추가 등.

In [66]:
# 1. 날짜 데이터에서 파생 변수 생성
df_date = pd.DataFrame({'P_Date': ['2026-03-8', '2026-3-2']})

df_date['P_Date'] =  pd.to_datetime(df_date['P_Date'])

df_date['Year'] = df_date['P_Date'].dt.year
df_date['Month'] = df_date['P_Date'].dt.month
df_date['Day'] = df_date['P_Date'].dt.day
df_date['DayOfWeek'] = df_date['P_Date'].dt.dayofweek  # 0 ~ 6 , 0:월, 6:일


# 2. 두 변수를 결합하여 새로운 인사이트 도출
df_sales = pd.DataFrame({'Total_Price': [10000, 25000], 'Items_Bought': [2, 5]})
df_sales['Price_per_Item'] = df_sales['Total_Price'] / df_sales['Items_Bought']

In [69]:
df_sales

,Total_Price,Items_Bought,Price_per_Item
0,10000,2,5000.0
1,25000,5,5000.0


In [67]:
df_date.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   P_Date     2 non-null      datetime64[ns]
 1   Year       2 non-null      int32         
 2   Month      2 non-null      int32         
 3   Day        2 non-null      int32         
 4   DayOfWeek  2 non-null      int32         
dtypes: datetime64[ns](1), int32(4)
memory usage: 180.0 bytes


In [68]:
df_date

,P_Date,Year,Month,Day,DayOfWeek
0,2026-03-08,2026,3,8,6
1,2026-03-02,2026,3,2,0
